# Programming Assignment 3 - Scalable Clustering Techniques

In [1]:
# imports for the whole homework
import os
os.environ["OMP_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from collections import defaultdict

from sklearn.cluster import KMeans

In [2]:
def load_covertype(file_path=r"./datasets/covtype.data.gz"):
    """
    Loads the Covertype dataset from a gzipped CSV-like file.
    Default path is relative to the current directory.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found at {file_path}. Please check the directory path.")

    print("Loading Covertype dataset")
    
    df = pd.read_csv(file_path, header=None, compression='gzip')
    
    X = df.iloc[:, :-1].values.astype(np.float32)
    
    # last column is the ground truth forest cover type (labels 1 through 7)
    y = df.iloc[:, -1].values.astype(np.int32)
    
    print(f"Dataset Loaded Successfully.")
    print(f"Instances: {X.shape[0]}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    
    return X, y

## 1. Lloyd’s algorithm for k-Means Clustering

As a baseline model implement
Lloyd’s algorithm [4] for k-means clustering and initialize it with the first k points as
initial cluster centers. The default convergence criteria is to stop the algorithm if none
of the cluster memberships have changed in comparison to the previous iteration.

□ Include a plot illustrating convergence of k-means.

□ Track the number of iterations needed for convergence and compare it to the other
implementations.

□ Report the achieved NMI averaged over at least 5 runs.

□ Report the runtime in [sec] for your algorithm averaged over at least 5 runs. Also
report the number of distance computations performed.

□ Briefly discuss your implementation of Lloyds algorithm.

In [ ]:
def lloyds_algorithm(X, k, y_true, max_iter=100):
    """
    Implementation of Lloyd's algorithm as seen in slide 9 of Unsupervised handout.
    """
    # initialization
    # mu_j^(0) = first k points of X
    mu = X[:k, :].copy()
    
    n_samples = X.shape[0]
    z = np.zeros(n_samples, dtype=int) # cluster memberships
    history_sq_dist = []

    total_dist_comps = 0
    
    start_time = time.time()

    for t in range(1, max_iter + 1):
        total_dist_comps += (n_samples * k)
        # cluster assignments (z_i^(t))
        # equation: z_i^(t) = argmin ||x_i - mu_j^(t-1)||_2^2. This is the literal computation, but it uses a lot of memory
        # distances_sq = np.sum((X[:, np.newaxis, :] - mu)**2, axis=2)

        # optimized calculation for large datasets, because the line from above takes too long, and it uses a lot of memory
        x_sq = np.sum(X**2, axis=1, keepdims=True)
        mu_sq = np.sum(mu**2, axis=1)
        distances_sq = x_sq + mu_sq - 2 * np.dot(X, mu.T)
        
        new_z = np.argmin(distances_sq, axis=1)
        
        # track the sum of squared distances for the convergence plot
        inertia = np.sum(np.min(distances_sq, axis=1))
        history_sq_dist.append(inertia)

        # convergence criteria - stop if none of the cluster memberships (z) have changed
        if np.array_equal(new_z, z):
            print(f"Converged at iteration {t}")
            break
        z = new_z.copy()

        # update cluster centers (mu_j^(t))
        # equation: mu_j^(t) = (1/n_j) * sum(x_i) for x_i assigned to j
        for j in range(k):
            assigned_points = X[z == j]
            if len(assigned_points) > 0:
                mu[j] = np.mean(assigned_points, axis=0)

    runtime = time.time() - start_time
    
    nmi = normalized_mutual_info_score(y_true, z, average_method='arithmetic')
    
    return nmi, t, history_sq_dist, runtime, total_dist_comps

In [ ]:
def run_experiment(X_raw, y_raw, k_values=[7, 30], num_runs=5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    final_report = {}

    for k in k_values:
        run_nmis = []
        run_iters = []
        run_times = []
        run_dist_comps = []
        
        print(f"\nEvaluating Lloyd's Algorithm for k={k}...")
        
        for r in range(num_runs):
            # shuffle data so first k points vary between runs for averaging
            X_s, y_s = shuffle(X_scaled, y_raw, random_state=r)
            
            nmi, iters, history, duration, dist_comps = lloyds_algorithm(X_s, k, y_s)
            
            run_nmis.append(nmi)
            run_iters.append(iters)
            run_times.append(duration)
            run_dist_comps.append(dist_comps)
            
            if r == num_runs-1:
                # plot illustrating convergence
                plt.figure(figsize=(7, 4))
                plt.plot(range(1, len(history) + 1), history, marker='x', color='tab:orange')
                plt.title(f"Lloyd's Convergence (k={k}, Run {r+1})")
                plt.xlabel("Iteration (t)")
                plt.ylabel("Total Squared Distance")
                plt.grid(True)
                plt.show()

        print(f"k={k} Average Results ({num_runs} runs):")
        print(f" - NMI: {np.mean(run_nmis):.4f}")
        print(f" - Iterations: {np.mean(run_iters):.1f}")
        print(f" - Runtime: {np.mean(run_times):.2f}s")
        print(f" - Distance Computations: {np.mean(run_dist_comps):,.0f}")
        
        final_report[k] = {"nmi": np.mean(run_nmis), "iters": np.mean(run_iters), "runtime": np.mean(run_times), "dist_comps": np.mean(run_dist_comps)}
        
    return final_report

In [ ]:
X, y = load_covertype()
result = run_experiment(X, y)
print("\nFinal Report:")
for k, metrics in result.items():
    print(f"k={k}: NMI={metrics['nmi']:.4f}, Avg Iterations={metrics['iters']:.1f}, Avg Runtime={metrics['runtime']:.2f}s, Avg Distance Computations={metrics['dist_comps']:,.0f}")

## 2. k-Means with Locality Sensitive Hashing (LSH)

Implement Lloyd’s algorithm using LSH to speed up the distance calculations. See the uploaded presentation
on Moodle on how LSH should be used.
Try out different settings in which you combine different hash functions with AND and
OR as it was discussed in the lecture. For example, you can have one setting where
you combine two hash functions with AND and in the second setting you combine two
functions with OR, but you could try out several combinations, with different hash
functions. Also try varying the number of buckets of your hashing function. Measure
the runtime timeLSH of your LSH implementation.

□ Report how you selected the parameters of LSH and how you combined your functions.

□ Report the accuracy using NMI and the runtime in seconds averaged over at least 5
runs. Also report the number of distance computations performed. If your implementa
tion doesn’t show a speed-up, discuss why this might be and also discuss whether this situation would change when working larger datasets.

□ Track the number of iterations needed for convergence (if it converges at all) and
compare it to the other implementations.

□ Briefly discuss your implementation of k-means with LSH.

In [ ]:
class LSHManager:
    def __init__(self, d, m_and, L_or, w):
        self.m_and = m_and
        self.L_or = L_or
        self.w = w
        self.a = np.random.normal(0, 1, (L_or, m_and, d))
        self.b = np.random.uniform(0, w, (L_or, m_and))

    def hash_points(self, X):
        projections = np.tensordot(self.a, X, axes=([2], [1])) + self.b[:, :, np.newaxis]
        hash_vals = np.floor(projections / self.w).astype(int)
        
        N = X.shape[0]
        hashed_data = []
        for l in range(self.L_or):
            # create a list of tuples for this specific table (band)
            table_hashes = [tuple(hash_vals[l, :, i]) for i in range(N)]
            hashed_data.append(table_hashes)
        return hashed_data

In [ ]:
def build_point_lookup(point_hashes, L_or):
    # lookup[table_index][hash_tuple] = [list of point indices]
    lookup = [defaultdict(list) for _ in range(L_or)]
    for l in range(L_or):
        for idx, h_tuple in enumerate(point_hashes[l]):
            lookup[l][h_tuple].append(idx)
    return lookup

In [ ]:
def assign_clusters_lsh_fast(X, mu, point_lookup, lsh_manager):
    N, k = X.shape[0], mu.shape[0]
    L = lsh_manager.L_or
    
    # initialize with infinity distance
    z = np.zeros(N, dtype=int)
    min_dists = np.full(N, np.inf)
    touched_by_lsh = np.zeros(N, dtype=bool)
    dist_comps = 0

    # hash the centers
    center_hashes = lsh_manager.hash_points(mu)

    # iterate through centers - slide 10: "Assign all points in this bucket to the center"
    for j in range(k):
        # a center has L different hash tuples (one per table)
        for l in range(L):
            c_h_val = center_hashes[l][j]
            # find points that fell into the same bucket as center j in table l
            colliding_point_indices = point_lookup[l].get(c_h_val, [])
            
            if colliding_point_indices:
                # calculate distance for these specific points only
                pts = X[colliding_point_indices]
                dists = np.sum((pts - mu[j])**2, axis=1)
                dist_comps += len(colliding_point_indices)
                
                # update assignments if this center is closer than previous candidates
                for idx_in_subset, original_idx in enumerate(colliding_point_indices):
                    if dists[idx_in_subset] < min_dists[original_idx]:
                        min_dists[original_idx] = dists[idx_in_subset]
                        z[original_idx] = j
                    touched_by_lsh[original_idx] = True

    # compute full distances for remaining points
    remaining_indices = np.where(~touched_by_lsh)[0]
    if len(remaining_indices) > 0:
        rem_X = X[remaining_indices]
        for i_idx, original_idx in enumerate(remaining_indices):
            dists = np.sum((rem_X[i_idx] - mu)**2, axis=1)
            best_j = np.argmin(dists)
            z[original_idx] = best_j
            min_dists[original_idx] = dists[best_j]
            dist_comps += k

    return z, np.sum(min_dists), dist_comps

In [ ]:
def lsh_algorithm(X, k, y_true, m_and=2, L_or=3, w=4.0, max_iter=100):
    lsh_manager = LSHManager(X.shape[1], m_and, L_or, w)
    
    # pre-hash points once
    point_hashes = lsh_manager.hash_points(X)
    point_lookup = build_point_lookup(point_hashes, L_or)
    
    mu = X[:k].copy()
    z = np.zeros(X.shape[0], dtype=int)
    history = []
    total_dist_comps = 0
    start_time = time.time()

    for t in range(1, max_iter + 1):
        new_z, sq_dist, comps = assign_clusters_lsh_fast(X, mu, point_lookup, lsh_manager)
        
        total_dist_comps += comps
        history.append(sq_dist)
        
        if np.array_equal(new_z, z):
            print(f"LSH Converged at t={t}")
            break
        z = new_z.copy()
        
        for j in range(k):
            mask = (z == j)
            if np.any(mask):
                mu[j] = np.mean(X[mask], axis=0)
                
    runtime = time.time() - start_time
    nmi = normalized_mutual_info_score(y_true, z, average_method='arithmetic')
    return nmi, t, history, runtime, total_dist_comps

In [ ]:
def run_experiment_lsh(X, y, k_values=[7, 30], num_runs=5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    for k in k_values:
        results_nmi = []
        results_iters = []
        results_dist = []
        results_time = []
        
        print(f"\n--- Running LSH k-Means for k={k} ---")
        
        for r in range(num_runs):
            # shuffle data so first k points vary between runs for averaging
            X_s, y_s = shuffle(X_scaled, y, random_state=r)
            
            nmi, t, history, runtime, dist_comps = lsh_algorithm(
                X_s, k, y_s, m_and=2, L_or=3, w=4.0
            )
            
            results_nmi.append(nmi)
            results_iters.append(t)
            results_dist.append(dist_comps)
            results_time.append(runtime)
            print(f"Run {r+1}: NMI={nmi:.4f}, Iters={t}, Distances={dist_comps:,.0f}, Time={runtime:.2f}s")

        print(f"\nFINAL AVERAGES FOR k={k}:")
        print(f"Avg NMI: {np.mean(results_nmi):.4f}")
        print(f"Avg Iterations: {np.mean(results_iters):.1f}")
        print(f"Avg Distance Computations: {np.mean(results_dist):,.0f}")
        print(f"Avg Runtime: {np.mean(results_time):.2f}s")

X, y = load_covertype(r"datasets\covtype.data.gz")
run_experiment_lsh(X, y)

## 3. k-means with coresets

Coresets are a compact representation of data sets,
such that models trained on a coreset are competitive with models trained on the full
dataset [3, 1]. In this task you will implement coresets for k-means clustering as in [1,
Algorithm 1]. For the the number of samples m, use 100, 1000, and 10000.

□ Report the runtime and NMI you achieve when using coresets of different size (as
described above) averaged over at least 5 runs. To do so, cluster the coresets using
sklearn’s k-means algorithm (you can supply sample weights to all needed functions).

□ Track the number of iterations needed for convergence and compare it to the other
implementations.

□ Analyze the variance of the accuracy obtained when using coresets for clustering by
computing 10 coresets for each choice of m.

In [3]:
def construct_lightweight_coreset(X, m):
    """
    Implementation of Algorithm 1 from Bachem et al. (2018).
    X: Scaled data (N, D)
    m: Coreset size
    """
    n_samples = X.shape[0]
    
    # 1. Compute global mean mu
    mu = np.mean(X, axis=0)
    
    # 2. Compute squared distances to mean: d(x, mu)^2
    dists_sq = np.sum((X - mu)**2, axis=1)
    sum_dists_sq = np.sum(dists_sq)
    
    # 3. Compute proposal distribution q(x)
    # q(x) = 0.5 * (1/n) + 0.5 * (d(x, mu)^2 / sum(d(x', mu)^2))
    q = 0.5 * (1.0 / n_samples) + 0.5 * (dists_sq / sum_dists_sq)
    
    # 4. Sample m indices according to q
    indices = np.random.choice(n_samples, size=m, p=q, replace=True)
    X_coreset = X[indices]
    
    # 5. Assign weights: w = 1 / (m * q(x))
    weights = 1.0 / (m * q[indices])
    
    return X_coreset, weights

def run_coreset_task(X_raw, y_raw, k_values=[7, 30], m_values=[100, 1000, 10000]):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    final_results = {}

    for k in k_values:
        print(f"\n{'='*40}\nEvaluating for k={k}\n{'='*40}")
        for m in m_values:
            nmis = []
            iters = []
            runtimes = []
            
            # Run at least 10 times to analyze variance as requested
            for i in range(10):
                start_time = time.time()
                
                # Construct coreset
                X_coreset, weights = construct_lightweight_coreset(X_scaled, m)
                
                # Use sklearn's KMeans with sample weights
                # n_init=1 and random_state to keep runs comparable to previous tasks
                kmeans = KMeans(n_clusters=k, init='random', n_init=1, max_iter=300)
                kmeans.fit(X_coreset, sample_weight=weights)
                
                # Predict on the FULL dataset to compute NMI
                full_assignments = kmeans.predict(X_scaled)
                
                end_time = time.time()
                
                nmi = normalized_mutual_info_score(y_raw, full_assignments, average_method='arithmetic')
                
                nmis.append(nmi)
                iters.append(kmeans.n_iter_)
                runtimes.append(end_time - start_time)
            
            print(f"Coreset size m={m}:")
            print(f"  - Avg NMI: {np.mean(nmis):.4f} (Std: {np.std(nmis):.4f})")
            print(f"  - Avg Iterations: {np.mean(iters):.1f}")
            print(f"  - Avg Runtime: {np.mean(runtimes):.4f}s")
            
            final_results[(k, m)] = nmis

    return final_results

X, y = load_covertype()
results = run_coreset_task(X, y)

Loading Covertype dataset
Dataset Loaded Successfully.
Instances: 581012, Features: 54, Classes: 7

Evaluating for k=7
Coreset size m=100:
  - Avg NMI: 0.1127 (Std: 0.0362)
  - Avg Iterations: 6.9
  - Avg Runtime: 0.2380s
Coreset size m=1000:
  - Avg NMI: 0.1444 (Std: 0.0388)
  - Avg Iterations: 11.3
  - Avg Runtime: 0.2304s
Coreset size m=10000:
  - Avg NMI: 0.1514 (Std: 0.0337)
  - Avg Iterations: 14.1
  - Avg Runtime: 0.2793s

Evaluating for k=30
Coreset size m=100:
  - Avg NMI: 0.1600 (Std: 0.0105)
  - Avg Iterations: 5.1
  - Avg Runtime: 0.2857s
Coreset size m=1000:
  - Avg NMI: 0.1827 (Std: 0.0063)
  - Avg Iterations: 13.3
  - Avg Runtime: 0.3121s
Coreset size m=10000:
  - Avg NMI: 0.1828 (Std: 0.0049)
  - Avg Iterations: 16.3
  - Avg Runtime: 0.3126s


## Report

Write a report about your work on this assignment, your findings and results. Make
sure to report all the information indicated above. Additionally report the following:

□ Show the performance in terms of NMI and runtime for the different approaches in
one plot or table.